# DataForge Arena - Safer GRPO Training on Colab

**Meta x PyTorch x Hugging Face x Scaler OpenEnv Hackathon 2026**

This notebook trains the DataForge surgeon policy with the safer T4 path now used in the repo:

- indexed schema and suspect columns in the prompt
- gentler late-stage anti-collapse shaping
- GRPO warmup enabled
- safer install flow with no `os.kill(...)`
- optional Hugging Face upload cell

Use a GPU runtime. On Colab, a T4 is the minimum supported target.


## 1. Setup and Install


In [ ]:
# Clone the repo, or refresh it if the runtime already has /content/dataforge-arena.
import os
import subprocess

REPO_DIR = '/content/dataforge-arena'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/vivekyarra/dataforge-arena.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')


In [ ]:
# Install a Colab-safe training stack.
# If torch is already imported in this runtime, restart the session once instead of force-killing it.
import os
import subprocess
import sys
from importlib.metadata import version

if 'torch' in sys.modules:
    raise SystemExit(
        'Torch is already loaded in this runtime. Use Runtime -> Restart session, then rerun the notebook from the top once.'
    )


def run(cmd, check=True):
    print('+', ' '.join(cmd))
    return subprocess.run(cmd, check=check)


run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])

exact_torch = [
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    'torch==2.10.0+cu128', 'torchvision==0.25.0+cu128', 'torchaudio==2.10.0+cu128',
]
fallback_torch = [
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    'torch>=2.9,<2.11', 'torchvision>=0.24,<0.26', 'torchaudio>=2.9,<2.11',
]

result = subprocess.run(exact_torch)
if result.returncode != 0:
    print('Exact torch pin unavailable in this runtime. Falling back to a compatible cu128 torch range.')
    run(fallback_torch)

run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    'unsloth==2026.4.8',
    'trl==0.24.0',
    'mergekit>=0.1.4',
    'peft>=0.14.0',
    'transformers',
    'datasets',
    'accelerate',
    'bitsandbytes',
    'openenv',
    'faker',
    'gradio>=4.26.0',
    'fastapi>=0.110.0',
    'uvicorn>=0.29.0',
    'python-multipart',
    'httpx',
    'matplotlib',
    'scipy',
    'huggingface_hub',
    'pydantic==2.12.5',
    'click==8.1.8',
])

import torch
import transformers.utils.hub
import unsloth

try:
    from trl import GRPOConfig, GRPOTrainer
except RuntimeError as exc:
    if 'mergekit' in str(exc):
        print('TRL requested mergekit at import time. Installing mergekit and retrying once.')
        run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'mergekit>=0.1.4'])
        from trl import GRPOConfig, GRPOTrainer
    else:
        raise

if not hasattr(transformers.utils.hub, 'TRANSFORMERS_CACHE'):
    transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv('HF_HOME', '/tmp/hf_cache')

print(f'torch={torch.__version__} | cuda={torch.version.cuda}')
print(f'trl={version("trl")} | mergekit={version("mergekit")} | unsloth={version("unsloth")} | pydantic={version("pydantic")} | click={version("click")}')
if version('click').startswith('8.2.'):
    print('WARNING: click 8.2.* can conflict with rasterio in some Colab runtimes.')
print('Dependency preflight passed.')
print('Note: a torchao warning about torch >= 2.11 is non-fatal here if TRL imports cleanly.')


In [ ]:
# Generate or refresh clean synthetic datasets used by training and evaluation.
!python training/generate_data.py


## 2. Verify GPU and Selected Model


In [ ]:
!nvidia-smi

import json
from training.model_config import detect_gpu, select_model, select_precision

gpu = detect_gpu()
model_cfg = select_model(gpu)
precision_cfg = select_precision(gpu)
print(json.dumps({'gpu': gpu, 'model': model_cfg, 'precision': precision_cfg}, indent=2))


## 3. Run Tests

This should stay green before training. The current repo target is `130 passed`.


In [ ]:
!python -m pytest -q


## 4. Prompt and Observation Sanity Check

This cell verifies the root-cause fix for flat reward: the model now sees schema indices directly in both `schema_str` and `_suspect_columns`.


In [ ]:
import json
import pandas as pd

from environment.corruptor import Corruptor
from environment.env import DataForgeEnv
from environment.schemas import HEALTHCARE_SCHEMA
from training.prompt import build_prompt

clean_df = pd.read_csv('data/healthcare_clean.csv')
env = DataForgeEnv(Corruptor(), HEALTHCARE_SCHEMA, clean_df)
obs = env.reset()
rows = json.loads(obs.rows_json)
prompt = build_prompt(obs)

print('Schema string:')
print(obs.schema_str)
print('---')
print('First ranked row:')
print(json.dumps(rows[0], indent=2) if rows else 'No rows returned')
print('---')
print(f'Estimated prompt tokens: {int(len(prompt) / 4)}')
assert '[' in obs.schema_str and ']' in obs.schema_str
if rows and rows[0]['_suspect_columns']:
    assert any(item.endswith(']') for item in rows[0]['_suspect_columns'])
print('Observation indexing looks correct.')


## 5. GRPO Training

The current T4 path uses:

- `200` max steps
- `4` generations per prompt
- `80` max completion tokens
- `warmup_ratio=0.08`
- gentler anti-collapse shaping that does not hammer exploration in the first 40 steps


In [ ]:
from training.model_config import detect_gpu, select_model

gpu = detect_gpu()
cfg = select_model(gpu)
seconds_per_step = 20 if gpu.get('vram_gb', 0) <= 16 else 12
print(f"Planned training config: {cfg['label']}")
print(f"Approx worst-case runtime: {(cfg['target_steps'] * seconds_per_step) / 60:.1f} minutes")

!python training/train_grpo.py


## 6. Judge-Facing Evaluation

This writes the trained checkpoint evaluation to `eval/results.json`.


In [ ]:
!python eval/evaluate.py --agent-mode grpo --model-path outputs/dataforge-surgeon --episodes 20 --tier 1 --steps 5 --seed 7

from pathlib import Path
print(Path('eval/results.json').read_text(encoding='utf-8'))


## 7. Training Curves and Diagnostics


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log = pd.read_csv('logs/training_log.csv')
fig, axes = plt.subplots(1, 4, figsize=(22, 4))

axes[0].plot(log['step'], log['total_reward'], color='#38bdf8', linewidth=2)
axes[0].set_title('Total Reward')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Reward')

axes[1].plot(log['step'], log['parse_success_rate'] * 100, color='#fbbf24', linewidth=2)
axes[1].set_title('Parse Success %')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('%')

axes[2].plot(log['step'], log['policy_shaping'], color='#34d399', linewidth=2)
axes[2].set_title('Policy Shaping')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Reward')

axes[3].plot(log['step'], log['dominant_tool_rate'] * 100, color='#fb7185', linewidth=2)
axes[3].set_title('Dominant Tool Rate %')
axes[3].set_xlabel('Step')
axes[3].set_ylabel('%')

plt.tight_layout()
plt.show()

print(f"Parse success first -> last: {log['parse_success_rate'].iloc[0] * 100:.1f}% -> {log['parse_success_rate'].iloc[-1] * 100:.1f}%")
print(f"Reward first -> last: {log['total_reward'].iloc[0]:+.3f} -> {log['total_reward'].iloc[-1]:+.3f}")
print(f"Accuracy delta mean: {log['accuracy_delta'].mean():+.4f}")


## 8. Save, Download, and Optional Hugging Face Upload


In [ ]:
import shutil
from pathlib import Path

export_dir = Path('colab_export')
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

for relative_path in [
    'outputs/dataforge-surgeon',
    'eval/results.json',
    'logs/training_log.csv',
    'logs/training_curve.png',
]:
    src = Path(relative_path)
    if not src.exists():
        continue
    dst = export_dir / src.name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

zip_path = shutil.make_archive('dataforge-arena-colab-export', 'zip', export_dir)
print(f'Created export zip: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print(f'Automatic download skipped: {exc}')


In [ ]:
# Optional: upload model artifacts and evidence to Hugging Face.
# Fill in HF_REPO_ID and set UPLOAD_TO_HF=True when you are ready.
import os
from pathlib import Path

from huggingface_hub import HfApi, create_repo, login, notebook_login

HF_REPO_ID = ''
UPLOAD_TO_HF = False
HF_TOKEN = os.getenv('HF_TOKEN', '')

if not UPLOAD_TO_HF:
    print('Skipping Hugging Face upload. Set UPLOAD_TO_HF=True and HF_REPO_ID to enable.')
else:
    if not HF_REPO_ID:
        raise ValueError('Set HF_REPO_ID before enabling UPLOAD_TO_HF.')

    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=True)
    else:
        notebook_login()

    model_dir = Path('outputs/dataforge-surgeon')
    if not model_dir.exists():
        raise FileNotFoundError('No local checkpoint found at outputs/dataforge-surgeon')

    create_repo(HF_REPO_ID, repo_type='model', exist_ok=True)
    api = HfApi()
    api.upload_folder(
        folder_path=str(model_dir),
        repo_id=HF_REPO_ID,
        repo_type='model',
        path_in_repo='dataforge-surgeon',
    )

    for local_path in ['eval/results.json', 'logs/training_log.csv', 'logs/training_curve.png']:
        path = Path(local_path)
        if path.exists():
            api.upload_file(
                path_or_fileobj=str(path),
                path_in_repo=local_path.replace('\\', '/'),
                repo_id=HF_REPO_ID,
                repo_type='model',
            )

    print(f'Upload complete: https://huggingface.co/{HF_REPO_ID}')
